# Function 3: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [ ]:
import numpy as np

input_data = np.load('initial_data/function_3/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.754364, 0.233177, 0.303208],
    [0.312229, 0.060777, 0.000904]
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


In [ ]:
output_data = np.load('initial_data/function_3/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    -0.09200841551496666,
    -0.18083748652026374,
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5]])
actual_output = -0.015979341188442648

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 3
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


In [ ]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


## Neural-network surrogate + input gradients

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Fit a small neural network to y. If the output scale is extreme, use log|y| instead.
# This cell automatically chooses log|y| if the ratio of magnitudes is very large or if values are near zero.
X = input_data.astype(np.float32)
y_raw = output_data.astype(np.float32)
use_log_abs = (np.nanmax(np.abs(y_raw)) / max(np.nanmin(np.abs(y_raw) + 1e-300), 1e-300) > 1e4)

y_target = np.log(np.abs(y_raw) + 1e-300).astype(np.float32) if use_log_abs else y_raw.copy()
target_name = "log_abs_y" if use_log_abs else "y"

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X).astype(np.float32)
y_scaled = y_scaler.fit_transform(y_target.reshape(-1, 1)).astype(np.float32).ravel()

X_t = torch.tensor(X_scaled, dtype=torch.float32)
y_t = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32)

class SurrogateNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        width = max(16, 4*d)
        self.net = nn.Sequential(
            nn.Linear(d, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, 1)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
model = SurrogateNN(d)
opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(3000):
    opt.zero_grad()
    pred = model(X_t)
    loss = loss_fn(pred, y_t)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_scaled = model(X_t).numpy().ravel()
    pred_target = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

print("Target modelled:", target_name)
print(f"In-sample MSE in target space: {mean_squared_error(y_target, pred_target):.6g}")
print(f"In-sample R² in target space: {r2_score(y_target, pred_target):.4f}")


In [ ]:
# Compute gradients of the network prediction with respect to original input variables.
X_grad = torch.tensor(X_scaled, dtype=torch.float32, requires_grad=True)
pred = model(X_grad)

# Sum is used so autograd gives one gradient per input point.
pred.sum().backward()
grad_scaled = X_grad.grad.detach().numpy()

# Convert from scaled-input / scaled-output gradient to original input scale.
grad_target = grad_scaled * (y_scaler.scale_[0] / x_scaler.scale_)
grad_norm = np.linalg.norm(grad_target, axis=1)

grad_df = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
grad_df["y"] = output_data
grad_df[target_name] = y_target
for j in range(d):
    grad_df[f"grad_x{j+1}"] = grad_target[:, j]
grad_df["grad_norm"] = grad_norm
grad_df["dominant_variable"] = [f"x{np.argmax(np.abs(row))+1}" for row in grad_target]

display(grad_df.sort_values("grad_norm", ascending=False))

avg_abs_grad = np.mean(np.abs(grad_target), axis=0)
print("Average absolute gradient:")
for j, val in enumerate(avg_abs_grad):
    print(f"x{j+1}: {val:.6g}")
print("Most influential variable on average:", f"x{np.argmax(avg_abs_grad)+1}")


In [ ]:
# Use the neural network to propose a point by searching random candidates.
# For minimisation, choose low predicted target. If target is log_abs_y, this finds small magnitude, not necessarily negative y.
rng = np.random.default_rng(2)
candidates = rng.random((30000 if d <= 4 else 60000, d)).astype(np.float32)
cand_scaled = x_scaler.transform(candidates).astype(np.float32)
with torch.no_grad():
    pred_scaled = model(torch.tensor(cand_scaled)).numpy().ravel()
    pred_target = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

nn_results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
nn_results[f"pred_{target_name}"] = pred_target
nn_results = nn_results.sort_values(f"pred_{target_name}", ascending=True)
display(nn_results.head(10))

best_nn = nn_results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(float)
print("NN suggested point:", np.round(best_nn, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_nn)))
